In [ ]:
import RF_Track as rft
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from pathlib import Path

from aux import plot_phase_space, plot_lattice
from data import parser
from structure_volume import create_lebt

%load_ext autoreload
%autoreload 2

### Initial Bunch

In [ ]:
B0 = parser()
B0T = rft.Bunch6dT(B0)
M0 = B0.get_phase_space('%x %xp %y %yp %N')

x  = M0[:,0]
xp = M0[:,1]
y  = M0[:,2]
yp = M0[:,3]
No = M0[:,4]


#plot_phase_space(x, xp, "x [mm]", "x' [rad]")
#plot_phase_space(y, yp, "y [mm]", "y' [rad]")
#plot_phase_space(x, y,  "x [mm]", "y [mm]")



### LEBT

In [ ]:
lebt = create_lebt(B0T)
lebt.dt_mm = 6      
lebt.tt_dt_mm = 50 

B1 = lebt.track(B0T)
I = B1.get_info()
    

print("\nX-X'")
print("emit =", I.emitt_x, "mm·mrad")
print("beta =", I.beta_x, "mm/mrad")
print("alpha =", I.alpha_x)

print("\nY-Y'")
print("emit =", I.emitt_y, "mm·mrad")
print("beta =", I.beta_y, "mm/mrad")
print("alpha =", I.alpha_y)

print("\n4D")
print("emit4D =", I.emitt_4d**2, "(mm·mrad)^2")


# Losses

#lost = lebt.get_lost_particles()
#X = lost[:, 0]
#Y = lost[:, 2]
#Z = lost[:, 4]
#
#print("X min/max:", X.min(), X.max())
#print("Y min/max:", Y.min(), Y.max())
#print("Z min/max:", Z.min(), Z.max())
#
#fig = plt.figure(figsize=(9, 7), dpi=120)
#ax3d = fig.add_subplot(111, projection="3d")
#
#sc = ax3d.scatter(
#    Z, X, Y,
#    c=Z,
#    cmap="viridis",
#    s=6,
#    alpha=0.7
#)
#
#ax3d.set_xlabel("Z (mm)")
#ax3d.set_ylabel("X (mm)")
#ax3d.set_zlabel("Y (mm)")
#ax3d.set_title("LEBT losses (3D)")
#fig.colorbar(sc, ax=ax3d, shrink=0.7, label="Z (mm)")
#
#plt.tight_layout()
#plt.show()

M1 = B1.get_phase_space("%X %xp %Y %yp")

x  = M1[:,0]
xp = M1[:,1]
y  = M1[:,2]
yp = M1[:,3]

#plot_phase_space(x, xp, "x [mm]", "x' [rad]")
#plot_phase_space(y,yp,"y [mm]", "y' [rad]")
#plot_phase_space(x, y,  "x [mm]", "y [mm]")


T = lebt.get_transport_table('%t %sigma_X %sigma_Y %N %mean_P %beta_x %beta_y %beta_z %alpha_x %alpha_y %alpha_z %mean_E %sigma_Px %sigma_Py')

t_rf  = T[:, 0]
mx = T[:, 1]
my = T[:, 2]
N  = T[:, 3]
P = T[:, 4]
beta_x = T[:, 5]
beta_y = T[:, 6]
beta_z = T[:, 7]
alpha_x = T[:, 8]
alpha_y = T[:, 9]
alpha_z = T[:, 10]
E_total = T[:, 11]          
sigma_px = T[:, 12]
sigma_py = T[:, 13]
mass_rf = 11174.9           
E = E_total - mass_rf     

# Convert ct to s

m = 12 * 931.494  # MeV/c^2
beta = P / np.sqrt(P**2 + m**2)
s = beta * t_rf   # mm
s = s / 1000      # → m

# Save transport data to a text file
out_file = Path("rftrack_vol_lebt_transport.txt")

data = np.column_stack((s, mx, my, N, P, beta_x, beta_y, beta_z, alpha_x, alpha_y, alpha_z, E, sigma_px, sigma_py))

np.savetxt(
    out_file,
    data,
    header="S[m] sigma_x[mm] sigma_y[mm] N mean_P[MeV/c] beta_x[mm/mrad] beta_y[mm/mrad] beta_z[mm/mrad] alpha_x[mrad] alpha_y[mrad] alpha_z[mrad] mean_E[MeV] sigma_px[mrad] sigma_py[mrad] ",
    fmt="%.10e"
)

results_dir = Path.cwd().resolve().parent / "beam_plots"
results_dir.mkdir(parents=True, exist_ok=True)

# Plot rms 

fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
color_x = 'tab:blue'
color_y = 'tab:red'
ax.plot(s,  mx,  color=color_x, linewidth=2.5, linestyle='-',  label=r'$\sigma_x$')
ax.plot(s, -mx,  color=color_x, linewidth=1.8, linestyle='-')
ax.plot(s,  my,  color=color_y, linewidth=2.5, linestyle='-',  label=r'$\sigma_y$')
ax.plot(s, -my,  color=color_y, linewidth=1.8, linestyle='-')
ax.set_xlabel('s [m]', fontsize=12)
ax.set_ylabel(r'$\sigma_{x,y}$ [mm]')

ax.set_xlim(0, 4.7)
ax.set_ylim(-max(mx)*1.2, max(mx)*1.2)

ax.legend(loc='upper left') 
ax.grid(True, linestyle='--', alpha=0.4)

plot_lattice(ax)

plt.tight_layout()
plt.show()

#Plot transmission

plt.figure(figsize=(10, 5), dpi=120)
plt.plot(s, N/N[0]*100, color='tab:red', linewidth=2.5, linestyle='-', label='Transmission')
plt.xlabel('s [m]', fontsize=12)
plt.ylabel('Transmission [%]', fontsize=12)
plt.xlim(0, 4.7) 
plt.ylim(0, 105)
plt.grid(True, linestyle='--', alpha=0.4)